# 材料消息与历史载体

所有输出目标、layer/head 均可选择。红色为幻觉，绿色为正常；完整计算没有峰筛选，表格仅展示保存的边。材料来源是 prompt V 边界归因，不等于原始事实身份或因果确认。

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import HTML, display
from experiments.reanchor_flow.message_lineage_report import render_sample

# 在 graph 仓库根目录运行；或把仓库根加入 sys.path。
OUTPUT = Path("experiments/reanchor_flow/outputs/attention_audit_v3/message_lineage")
index = json.loads((OUTPUT / "index.json").read_text())
AUDIT = Path(index["native_audit"])  # 下载后可改为本机原始 compact NPZ/labels 目录
entries = index["samples"]
[(e["split"], e["task_type"], e["sample_id"]) for e in entries[:20]]

In [ ]:
SPLIT, TASK, SAMPLE = entries[0]["split"], entries[0]["task_type"], entries[0]["sample_id"]
LAYER, HEAD = 1, 0
e = next(e for e in entries if (e["split"], e["task_type"], e["sample_id"]) == (SPLIT, TASK, SAMPLE))
page = render_sample(AUDIT / e["path"], OUTPUT / e["path"], LAYER, HEAD)
display(HTML(page.read_text()))

In [ ]:
import matplotlib.pyplot as plt
QUERY_SLOT = 0  # q = row_position[QUERY_SLOT]；预测 token q+1
with np.load(AUDIT / e["path"]) as f: trace = dict(f)
with np.load(OUTPUT / e["path"]) as f: flow = dict(f)
with np.load((AUDIT / e["path"]).with_suffix(".labels.npz")) as f: labels = f["labels"]
q = int(trace["row_position"][QUERY_SLOT])
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(trace["residual_margin"][:, QUERY_SLOT], label="Native residual readout")
ax.plot(flow["material_residual_margin"][:, QUERY_SLOT], label="Attributed material component")
ax.axhline(0, color="gray", lw=.7)
ax.set(title=f"q={q} predicts token {q+1}, label={labels[QUERY_SLOT]}", xlabel="Residual stage", ylabel="Signed observed-vs-runner margin")
ax.legend(); plt.show()
# 三种分量共享同一目标方向；不能把该图称为事实语义读出。